# Seed comparison: synthetic vs LLM vs ESConv

Purpose: run a tiny grid to compare the three seeding modes before choosing one for larger Phase 3 runs.

- **Synthetic seeds**: fixed templates per emotion (controlled, but less realistic).
- **LLM seeds**: generated per trial (more variety; can drift or be empty if model is flaky).
- **ESConv seeds**: sampled utterances from ESConv, classifier-checked to match target emotion (most realistic; requires ESConv download).

Outputs: three CSVs/heatmaps under `results/seed_comparison_*` plus meta/log files.

In [1]:
!pip uninstall -y dynamic-conversation dynamic_conversation
!pip cache purge
!pip install --no-cache-dir git+https://github.com/Javin-Mendiratta/Dynamic-Conversation.git@derek_12_13


from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OpenAI')

from pathlib import Path
import json
from datasets import load_dataset
from dynamic_conversation import (
    SingleTurnSimulator,
    SimulationConfig,
    ResponseStrategy,
    build_esconv_seed_bank,
)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# Shared config
cfg = SimulationConfig(
    model="gpt-4o-mini",
    max_tokens=800,
    temperature=1.0,
    brevity_hint="Reply in 1-2 sentences, keep emotion visible."
)

emotions = ["anger", "joy"]
strategies = [ResponseStrategy.VALIDATE, ResponseStrategy.GUIDE]
style_mod = "concise and emotionally attuned"

# Build ESConv seed bank (classifier-checked into DistilRoBERTa labels)
esconv = load_dataset("thu-coai/esconv")
seed_bank = build_esconv_seed_bank(esconv, emotions=emotions, per_emotion=6, max_chars=200, use_gpu=False)
seed_bank

Found existing installation: dynamic-conversation 0.1.1
Uninstalling dynamic-conversation-0.1.1:
  Successfully uninstalled dynamic-conversation-0.1.1
Files removed: 12
  Cloning https://github.com/Javin-Mendiratta/Dynamic-Conversation.git (to revision derek_12_13) to /tmp/pip-req-build-j5b4otud
  Running command git clone --filter=blob:none --quiet https://github.com/Javin-Mendiratta/Dynamic-Conversation.git /tmp/pip-req-build-j5b4otud
  Running command git checkout -b derek_12_13 --track origin/derek_12_13
  Switched to a new branch 'derek_12_13'
  Branch 'derek_12_13' set up to track remote branch 'derek_12_13' from 'origin'.
  Resolved https://github.com/Javin-Mendiratta/Dynamic-Conversation.git to commit 46cc0ccf14de6c352f0406feedfbfafbb0b4c1cf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for dynamic-conversation: filename=dynamic_conversation-0.1.1-py3-none-any.whl size=24153

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading emotion classifier: j-hartmann/emotion-english-distilroberta-base


Device set to use cpu


{'anger': ['I am sooo pissed at my ex.',
  "I can't take it anymore the stress from my parents. They don't acknowledge I am doing my best.",
  'Hey! How can I help?',
  'I am very upset',
  'How could my partner say he loves me, go out with me for one year and then cheat.',
  'My board is going around me to my staff and I am frustrated.'],
 'joy': ['Hello good afternoon.',
  'Hi, are you having a good day at the moment?',
  'Hi!',
  'Good evening, how are you feeling at the moment?',
  'Good morning! What can I help you with today?',
  'Hello there! How can I encourage you today?']}

In [2]:
def run_and_report(suffix, use_llm_seed=False, use_esconv_seed=False, esconv_seeds=None):
    sim = SingleTurnSimulator(
        use_gpu=False,
        config=cfg,
        prompt_for_key=True,
        esconv_seeds=esconv_seeds,
    )
    df = sim.run_batch(
        emotions=emotions,
        strategies=strategies,
        runs_per_pair=1,
        style_modifier=style_mod,
        use_llm_seed=use_llm_seed,
        include_baseline=True,
        use_esconv_seed=use_esconv_seed,
        save_csv=results_dir / f"seed_comparison_{suffix}.csv",
        save_heatmap=results_dir / f"seed_comparison_{suffix}_heatmap.png",
    )
    meta_path = results_dir / f"seed_comparison_{suffix}.meta.json"
    log_path = results_dir / f"seed_comparison_{suffix}.log"
    meta = json.load(open(meta_path)) if meta_path.exists() else {}
    log = log_path.read_text() if log_path.exists() else ""
    return df, meta, log

# Synthetic seeds (default)
df_syn, meta_syn, log_syn = run_and_report("synthetic", use_llm_seed=False, use_esconv_seed=False)

# LLM seeds
df_llm, meta_llm, log_llm = run_and_report("llm", use_llm_seed=True, use_esconv_seed=False)

# ESConv seeds
df_esconv, meta_esconv, log_esconv = run_and_report("esconv", use_llm_seed=False, use_esconv_seed=True, esconv_seeds=seed_bank)


Loading emotion classifier: j-hartmann/emotion-english-distilroberta-base


Device set to use cpu


Loading emotion classifier: j-hartmann/emotion-english-distilroberta-base


Device set to use cpu


Loading emotion classifier: j-hartmann/emotion-english-distilroberta-base


Device set to use cpu


In [4]:
print("Synthetic meta:", meta_syn)
print("Synthetic log:\n", log_syn)

print("LLM meta:", meta_llm)
print("LLM log:\n", log_llm)

print("ESConv meta:", meta_esconv)
print("ESConv log:\n", log_esconv)



Synthetic meta: {'emotions': ['anger', 'joy'], 'strategies': ['Validate', 'Guide', 'baseline'], 'runs_per_pair': 1, 'style_modifier': 'concise and emotionally attuned', 'use_llm_seed': False, 'include_baseline': True, 'config': {'model': 'gpt-4o-mini', 'temperature': 1.0, 'max_tokens': 800, 'retries': 3, 'backoff_seconds': 2.0}, 'success_count': 6, 'failure_count': 0}
Synthetic log:
 
LLM meta: {'emotions': ['anger', 'joy'], 'strategies': ['Validate', 'Guide', 'baseline'], 'runs_per_pair': 1, 'style_modifier': 'concise and emotionally attuned', 'use_llm_seed': True, 'include_baseline': True, 'config': {'model': 'gpt-4o-mini', 'temperature': 1.0, 'max_tokens': 800, 'retries': 3, 'backoff_seconds': 2.0}, 'success_count': 6, 'failure_count': 0}
LLM log:
 
ESConv meta: {'emotions': ['anger', 'joy'], 'strategies': ['Validate', 'Guide', 'baseline'], 'runs_per_pair': 1, 'style_modifier': 'concise and emotionally attuned', 'use_llm_seed': False, 'include_baseline': True, 'config': {'model': 'g

In [9]:
# LLM SEEDING (we simply prompt it to express something with a certain emotion)
df_llm

,intended_emotion,seed_text,seed_emotion_detected,seed_confidence,strategy,style_modifier,strategy_reply,followup_reply,followup_emotion,followup_confidence,status
0,anger,I can't believe you took my things without ask...,anger,0.669816,Validate,concise and emotionally attuned,It sounds like you're feeling really hurt and ...,"I appreciate you acknowledging my feelings, bu...",anger,0.383169,success
1,anger,I can't believe they did that! It's absolutely...,anger,0.884149,Guide,concise and emotionally attuned,I completely understand how frustrating that m...,"I appreciate the advice, but honestly, I’m so ...",neutral,0.517683,success
2,anger,I can't believe you did that after everything ...,anger,0.934525,baseline,concise and emotionally attuned,I understand why you feel hurt and frustrated....,Listening isn't enough right now. You need to ...,anger,0.391597,success
3,joy,I can’t believe it—I just got the job I’ve alw...,surprise,0.871780,Validate,concise and emotionally attuned,That’s incredible! It sounds like you’re feeli...,Thank you! I really am on cloud nine—it's like...,joy,0.829369,success
4,joy,I just found out I got the job! I can't believ...,surprise,0.599646,Guide,concise and emotionally attuned,That’s amazing—congratulations! Take a moment ...,Thank you so much! I absolutely will—celebrati...,joy,0.992115,success
5,joy,I can’t believe it! This is the happiest momen...,surprise,0.502940,baseline,concise and emotionally attuned,That’s amazing! I’m so thrilled for you—this j...,Thank you so much! I can't stop smiling; it fe...,joy,0.977548,success


In [7]:
df_syn

,intended_emotion,seed_text,seed_emotion_detected,seed_confidence,strategy,style_modifier,strategy_reply,followup_reply,followup_emotion,followup_confidence,status
0,anger,I am furious right now about how unfair this a...,anger,0.985550,Validate,concise and emotionally attuned,It sounds like you're really feeling the weigh...,"Absolutely, it's like no matter what I do, the...",anger,0.902335,success
1,anger,Everything went wrong today and I am boiling w...,anger,0.989366,Guide,concise and emotionally attuned,I’m really sorry to hear that today has been s...,"I appreciate the suggestion, but right now, it...",fear,0.341104,success
2,anger,I am furious right now about how unfair this a...,anger,0.985550,baseline,concise and emotionally attuned,I completely understand why you feel that way....,"I appreciate that, but being understood doesn'...",anger,0.960067,success
3,joy,I'm excited and joyful about this news!,joy,0.993843,Validate,concise and emotionally attuned,It sounds like you’re feeling really happy and...,Absolutely! It truly feels wonderful to share ...,joy,0.988653,success
4,joy,I'm excited and joyful about this news!,joy,0.993843,Guide,concise and emotionally attuned,That's wonderful to hear! Why not celebrate to...,That's such a great idea! I'm particularly thr...,joy,0.944769,success
5,joy,I feel genuinely happy about how things turned...,joy,0.994545,baseline,concise and emotionally attuned,That’s wonderful to hear! It’s great to see th...,Thank you! It really means a lot to have suppo...,joy,0.981127,success


In [8]:
df_esconv

,intended_emotion,seed_text,seed_emotion_detected,seed_confidence,strategy,style_modifier,strategy_reply,followup_reply,followup_emotion,followup_confidence,status
0,anger,I am sooo pissed at my ex.,anger,0.851917,Validate,concise and emotionally attuned,It sounds like you're really overwhelmed and f...,"Yeah, I am overwhelmed! It's just infuriating ...",surprise,0.725419,success
1,anger,"How could my partner say he loves me, go out w...",anger,0.640565,Guide,concise and emotionally attuned,I’m really sorry you’re feeling this way; it’s...,It’s infuriating that he could betray my trust...,anger,0.958772,success
2,anger,My board is going around me to my staff and I ...,anger,0.946093,baseline,concise and emotionally attuned,I can see how frustrating that must be for you...,It's just infuriating to feel sidelined like t...,anger,0.954730,success
3,joy,Hi!,joy,0.763597,Validate,concise and emotionally attuned,It sounds like you're feeling a bit uncertain ...,Absolutely! I love connecting and sharing thou...,joy,0.987289,success
4,joy,Good morning! What can I help you with today?,joy,0.719662,Guide,concise and emotionally attuned,Good morning! You could ask about their day or...,That's such a great idea! How's your morning g...,joy,0.598781,success
5,joy,"Hi, are you having a good day at the moment?",joy,0.517034,baseline,concise and emotionally attuned,"Hi! Thanks for asking. I'm doing well, how abo...",I’m so glad to hear you’re doing well! My day’...,joy,0.987139,success
